# 04. LangGraph 로 만든 마피아 게임

**목적**: 03에서 `GameEngine`의 while 문으로 짠 게임 진행을 **LangGraph(상태 그래프)**로 다시 표현한다.
규칙 판정은 `mafia_core`의 함수를 그대로 재사용하고, 그래프는 "다음에 무엇을 실행할지"만 맡는다.

## LangChain `create_agent` vs LangGraph

| | create_agent (06_Agent) | LangGraph |
|---|---|---|
| 관점 | 한 명의 AI가 도구를 골라 쓰는 **개별 판단** | 게임 전체의 **진행 순서와 상태 관리** |
| 흐름 | LLM이 정함 (도구 호출 반복) | 개발자가 노드와 엣지로 정함 |
| 분기 | 없음 | 조건부 엣지 (승리 여부 등) |
| 사람 개입 | 어려움 | `interrupt` / `Command(resume=...)` |
| 상태 저장 | 없음 | checkpointer |

이 프로젝트에서는 **AI 한 명의 두뇌 = create_agent, 게임 심판/진행 = LangGraph**로 역할을 나눴다.

## LangGraph 핵심 개념 (이 노트북에서 순서대로 사용)

1. **State**: 노드들이 함께 읽고 쓰는 공유 데이터 (`TypedDict`). 우리는 `game`(GameState), 밤 선택, 발언/투표 순서 등을 담는다.
2. **Node**: State를 받아 일을 하고 **바뀐 부분만 딕셔너리로 돌려주는 함수**. (`night_mafia`, `morning`, `speak` ...)
3. **Edge**: 노드 사이의 고정된 다음 길. (`night_start -> night_mafia`)
4. **Conditional edge**: 함수의 반환값에 따라 다음 노드를 고른다. (승부가 났으면 `ending`, 아니면 다음 단계)
5. **interrupt / Command(resume)**: 노드 실행 중 그래프를 멈추고 사람의 입력을 기다린다. 입력을 받아 재개한다.
6. **Checkpointer(MemorySaver) + thread_id**: 노드가 끝날 때마다 State를 저장한다. `thread_id`는 "이 게임"의 이름표이다. interrupt는 체크포인터가 있어야 동작한다.

In [1]:
# ===== 0. 작업 폴더 맞추기 =====
# 이 노트북은 08_Mafia_Agent 폴더에서 실행한다고 가정한다.
# (VS Code에서 상위 폴더를 열었을 때도 동작하도록 폴더를 찾아 이동한다.)
import os
import sys
from pathlib import Path

if not Path("config.py").exists():
    if Path("08_Mafia_Agent/config.py").exists():
        os.chdir("08_Mafia_Agent")
    else:
        raise FileNotFoundError("08_Mafia_Agent 폴더에서 노트북을 열어주세요.")

sys.path.insert(0, os.getcwd())

import config

print("===== 작업 폴더 =====")
print(config.BASE_DIR)
print("LLM:", config.LLM_MODEL, "/ 임베딩:", config.EMBED_MODEL)

===== 작업 폴더 =====
.
LLM: qwen3:8b / 임베딩: bge-m3


In [2]:
# ===== 1. 모듈 불러오기 =====
import random
import time

from langgraph.types import Command

import rag
import mafia_core as core
import mafia_agents as agents
import mafia_graph as mg

vs = rag.load_db()
print("===== 준비 완료 =====")

===== 준비 완료 =====


## 1. 그래프 구조

게임 한 판의 흐름이다. 밤은 마피아(제안, 최종) -> 의사 -> 경찰 순서로 노드를 나눴고, 발언과 투표는 **한 명씩 한 번의 노드 실행**으로 처리한다.
(`speak`, `vote` 노드는 자기 자신으로 돌아오는 **반복 엣지**로 모든 사람을 처리한다.)

노드 이름과 역할

| 노드 | 하는 일 (mafia_core 재사용) |
|---|---|
| setup | 게임 시작 안내 |
| night_start / night_mafia_first / night_mafia_second / night_doctor / night_police | 밤 시작, 첫 마피아(제안) -> 둘째 마피아(최종) -> 의사 -> 경찰 선택 수집 |
| morning | 밤 결과 판정(`apply_night`) + 승리 판정 |
| day_start / speak | 낮 토론 순서 정하기 / 한 명 발언 |
| vote_start / vote / tally | 투표 시작 / 한 명 투표 / 개표 (동점이면 재투표) |
| execution | 처형 후 승리 판정 |
| ending | 종료 처리 |

조건부 엣지: `morning` 뒤 (승부 -> ending / 계속 -> day_start), `speak` 반복, `vote` 반복, `tally` 뒤 (동점 -> vote 재투표 / execution), `execution` 뒤 (승부 -> ending / 계속 -> night_start)

In [3]:
# ===== 2. 그래프 시각화 =====
# 먼저 LLM 없이 구조만 보기 위해 RandomController 로 그래프를 만든다.
game0 = core.new_game(list(config.CHARACTERS), None, seed=0)
random_ctrls = {p.name: core.RandomController(p.name, seed=i) for i, p in enumerate(game0.players)}

graph = mg.build_graph(random_ctrls)

# PNG 는 인터넷(mermaid.ink)이 필요하므로, 실패하면 Mermaid 텍스트를 출력한다.
mermaid_text = mg.show_graph(graph)

===== 그래프 PNG 생성 완료 =====
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	setup(setup)
	night_start(night_start)
	night_mafia_first(night_mafia_first)
	night_mafia_second(night_mafia_second)
	night_doctor(night_doctor)
	night_police(night_police)
	morning(morning)
	day_start(day_start)
	speak(speak)
	vote_start(vote_start)
	vote(vote)
	tally(tally)
	execution(execution)
	ending(ending)
	__end__([<p>__end__</p>]):::last
	__start__ --> setup;
	day_start --> speak;
	execution -.-> ending;
	execution -.-> night_start;
	morning -.-> day_start;
	morning -.-> ending;
	night_doctor --> night_police;
	night_mafia_first --> night_mafia_second;
	night_mafia_second --> night_doctor;
	night_police --> morning;
	night_start --> night_mafia_first;
	setup --> night_start;
	speak -.-> vote_start;
	tally -.-> execution;
	tally -.-> vote;
	vote -.-> tally;
	vote_start --> vote;
	ending --> __end__;
	speak -.-> speak;
	vote -.-> vote;
	classDef default fill

## 2. 빠른 동작 확인 (LLM 없이, RandomController)

LLM을 쓰기 전에 그래프 흐름이 맞는지 확인한다. `run_spectator`는 사람 입력 없이 끝까지 실행한다.
`thread_id`는 체크포인트 저장소에서 이 게임을 구분하는 이름이다.

In [4]:
# ===== 3. RandomController 로 그래프 한 판 =====
final = mg.run_spectator(graph, game0, thread_id="quick-test")

print("\n===== 그래프 실행 완료 =====")
print("승자:", final.winner, "| 진행 일수:", final.day)
print("체크포인트 개수:", len(list(graph.get_state_history(mg.make_config("quick-test")))))


===== 마피아 게임 시작 (8명) =====
참가자: 강태호, 윤서진, 박준수, 최민지, 정도현, 한소율, 오지환, 임채원
  [비밀 · 시민 강태호] 당신의 직업: 시민
  [비밀 · 마피아 윤서진] 당신의 직업: 마피아
  [비밀 · 마피아 윤서진] 동료 마피아: 정도현
  [비밀 · 시민 박준수] 당신의 직업: 시민
  [비밀 · 경찰 최민지] 당신의 직업: 경찰
  [비밀 · 마피아 정도현] 당신의 직업: 마피아
  [비밀 · 마피아 정도현] 동료 마피아: 윤서진
  [비밀 · 의사 한소율] 당신의 직업: 의사
  [비밀 · 시민 오지환] 당신의 직업: 시민
  [비밀 · 시민 임채원] 당신의 직업: 시민

===== 1일차 밤 =====
  [비밀 · 마피아 윤서진] 박준수를 노립니다 (이유: 무작위 선택) [제안]
  [비밀 · 마피아 정도현] 박준수를 노립니다 (이유: 무작위 선택) [최종]
  [비밀 · 의사 한소율] 정도현 보호
박준수(시민)이 밤에 사망
  [비밀 · 경찰 최민지] 윤서진 조사 결과: 마피아다

===== 1일차 낮 토론 =====
[윤서진] 어젯밤 일이 마음에 걸리네요.
[강태호] 어젯밤 일이 마음에 걸리네요.
[한소율] 저는 아직 잘 모르겠어요.
[오지환] 저는 아직 잘 모르겠어요.
[임채원] 저는 아직 잘 모르겠어요.
[최민지] 저는 아직 잘 모르겠어요.
[정도현] 저는 아직 잘 모르겠어요.

===== 1일차 투표 =====
[투표] 강태호 -> 한소율
[투표] 윤서진 -> 한소율
[투표] 최민지 -> 오지환
[투표] 정도현 -> 강태호
[투표] 한소율 -> 정도현
[투표] 오지환 -> 윤서진
[투표] 임채원 -> 최민지
한소율(의사)가 투표로 처형됨

===== 2일차 밤 =====
  [비밀 · 마피아 윤서진] 임채원을 노립니다 (이유: 무작위 선택) [제안]
  [비밀 · 마피아 정도현] 강태호를 노립니다 (이유: 무작위 선택) [최종]
강태호(시민)이 밤에 사망
  [비밀 · 경찰 최민지] 강태호 조사 결과

### 체크포인트 들여다보기

노드가 하나 끝날 때마다 State가 저장된다. `get_state_history`로 지나온 노드 순서를 볼 수 있다.

In [5]:
# ===== 4. 실행 경로 확인 =====
cfg = mg.make_config("quick-test")
history = list(graph.get_state_history(cfg))

path = [snap.next[0] for snap in reversed(history) if snap.next]

print("실행된 노드 순서(앞 25개):", path[:25])
print("총 노드 실행 횟수:", len(path))

실행된 노드 순서(앞 25개): ['__start__', 'setup', 'night_start', 'night_mafia_first', 'night_mafia_second', 'night_doctor', 'night_police', 'morning', 'day_start', 'speak', 'speak', 'speak', 'speak', 'speak', 'speak', 'speak', 'vote_start', 'vote', 'vote', 'vote', 'vote', 'vote', 'vote', 'vote', 'tally']
총 노드 실행 횟수: 47


## 3. 관전 모드: LLM 8명을 LangGraph 로 실행

03과 같은 `LLMPlayer`를 컨트롤러로 넣기만 하면 된다. **진행(그래프)과 판단(LLM)이 분리**되어 있어서 컨트롤러만 바꿔 끼울 수 있다.
수 분이 걸린다.

In [6]:
# ===== 5. LLM 관전 모드 (LangGraph) =====
SEED = 21

game = core.new_game(list(config.CHARACTERS), None, seed=SEED)
ctrls = agents.build_controllers(game, vs)
llm_graph = mg.build_graph(ctrls)

t0 = time.time()
final = mg.run_spectator(llm_graph, game, thread_id="spectator-%d" % SEED)
print("\n승자:", final.winner, "| 일수:", final.day, "| 소요 %.0f초" % (time.time() - t0))
print(core.reveal_report(final))


===== 마피아 게임 시작 (8명) =====
참가자: 강태호, 윤서진, 박준수, 최민지, 정도현, 한소율, 오지환, 임채원
  [비밀 · 시민 강태호] 당신의 직업: 시민
  [비밀 · 마피아 윤서진] 당신의 직업: 마피아
  [비밀 · 마피아 윤서진] 동료 마피아: 박준수
  [비밀 · 마피아 박준수] 당신의 직업: 마피아
  [비밀 · 마피아 박준수] 동료 마피아: 윤서진
  [비밀 · 시민 최민지] 당신의 직업: 시민
  [비밀 · 시민 정도현] 당신의 직업: 시민
  [비밀 · 시민 한소율] 당신의 직업: 시민
  [비밀 · 의사 오지환] 당신의 직업: 의사
  [비밀 · 경찰 임채원] 당신의 직업: 경찰

===== 1일차 밤 =====
  [비밀 · 마피아 윤서진] 최민지를 노립니다 (이유: 최민지가 논리적으로 추리하는 모습을 보이며 시민팀에 위협이 될 수 있는 인물로 판단했습니다.) [제안]
  [비밀 · 마피아 박준수] 최민지를 노립니다 (이유: 논리적으로 추리하며 시민팀에 위협이 될 수 있는 인물로 판단했습니다.) [최종]
  [비밀 · 의사 오지환] 오지환 보호
최민지(시민)이 밤에 사망
  [비밀 · 경찰 임채원] 박준수 조사 결과: 마피아다

===== 1일차 낮 토론 =====
[오지환] 강태호야, 왜 그런지 말해줘! 느낌이 와!
[강태호] 야, 오지환. 너 아까 왜 나한테 그런 말했냐? 난 지금까지 아무것도 안 했는데.
[박준수] 에이~ 근데 강태호, 왜 오지환한테 그런 말했냐? 낯이 가려서 그런 거 아니야? 진짜 웃으면서 말하는데 눈은 안 웃는 거 같아ㅋㅋ
[윤서진] 오지환의 감정 표현이 과도하며, 강태호가 그에게 그런 말을 했는데 그 이유가 명확하지 않습니다. 박준수는 웃음이 어울리지 않는 행동을 보였습니다. 이에 따라 오지환과 강태호를 의심합니다.
[한소율] 강태호 씨, 오지환 씨에게 그런 말을 했던 이유는 혹시 어떤 이유가 있으신가요?
[정도현] 강태호, 오지환한테 왜 그런 말했냐? 왜 그런 건지 말해.
[임채원] 박준수는

## 4. Human-in-the-loop: 사람 입력 대기와 재개

참여 모드에서 사람 차례가 오면 노드 안의 `interrupt(payload)`가 실행되어 **그래프가 멈춘다**.
호출자는 `result["__interrupt__"]`로 질문 내용(payload)을 받고, `Command(resume=답)`으로 다시 `invoke`하면 멈춘 곳에서 이어진다.

```
graph.invoke(입력, cfg)                 -> 사람 차례에서 멈춤, result["__interrupt__"] 에 질문
graph.invoke(Command(resume="답"), cfg) -> 같은 thread_id 의 체크포인트에서 재개
```

> 주의: 재개하면 멈췄던 **노드를 처음부터 다시 실행**한다. 그래서 `speak`/`vote`를 "한 명 = 한 노드"로 쪼갰다.
> (한 노드에서 AI 7명을 모두 처리하면 사람 차례마다 AI 호출이 반복된다.)
>
> 같은 이유로 마피아도 `night_mafia_first`(제안) / `night_mafia_second`(최종)로 나눠 첫 제안을 State(`proposal`)에 저장한다. 사람이 둘째 마피아일 때 재개해도 첫 마피아 AI는 다시 호출되지 않는다.
>
> 또한 `mafia_core`의 안전장치가 예외를 삼키기 때문에, `GraphHumanController`는 interrupt가 던지는 예외를 `BaseException`으로 감싸 노드 밖으로 통과시킨다 (`mafia_graph.py` 참고).

먼저 **스크립트로 자동 응답**하는 시연으로 흐름을 눈으로 확인한다. (AI는 RandomController라서 빠르다)

In [7]:
# ===== 6. interrupt / resume 단계별 시연 =====
ai7 = random.Random(5).sample(config.CHARACTERS, 7)
hgame = core.new_game(ai7, config.HUMAN_NAME, seed=5)
hctrls = {p.name: core.RandomController(p.name, seed=i) for i, p in enumerate(hgame.players)}
hctrls[config.HUMAN_NAME] = mg.GraphHumanController(config.HUMAN_NAME)   # 사람 자리만 interrupt 컨트롤러

hgraph = mg.build_graph(hctrls, on_event=lambda ev: None)   # 이벤트 출력 생략
hcfg = mg.make_config("hitl-demo")

result = hgraph.invoke({"game": hgame}, hcfg)

for step in range(4):
    if "__interrupt__" not in result:
        break
    payload = result["__interrupt__"][0].value
    print("[멈춤 %d] 종류=%s | 다음 실행 노드=%s" % (step + 1, payload["kind"], hgraph.get_state(hcfg).next))
    print("   질문:", payload["prompt"], "| 후보:", payload["candidates"][:3], "...")
    answer = "시연용 발언입니다." if payload["kind"] == "speak" else payload["candidates"][0]
    print("   재개 값:", answer)
    result = hgraph.invoke(Command(resume=answer), hcfg)

[멈춤 1] 종류=speak | 다음 실행 노드=('speak',)
   질문: 발언을 입력하세요. (빈 입력은 패스) | 후보: ['정도현', '박준수', '임채원'] ...
   재개 값: 시연용 발언입니다.
[멈춤 2] 종류=vote | 다음 실행 노드=('vote',)
   질문: 투표할 대상을 고르세요. | 후보: ['정도현', '박준수', '임채원'] ...
   재개 값: 정도현
[멈춤 3] 종류=vote | 다음 실행 노드=('vote',)
   질문: 투표할 대상을 고르세요. | 후보: ['강태호', '정도현'] ...
   재개 값: 강태호


In [8]:
# ===== 7. 나머지는 자동 응답으로 끝까지 =====
def auto_answer(payload):
    if payload["kind"] == "speak":
        return "저는 시민이라고 생각합니다."
    return random.choice(payload["candidates"])


while "__interrupt__" in result:
    payload = result["__interrupt__"][0].value
    result = hgraph.invoke(Command(resume=auto_answer(payload)), hcfg)

hfinal = mg.get_game(hgraph, hcfg)
print("===== 참여 모드(스크립트) 종료 =====")
print("승자:", hfinal.winner, "| 일수:", hfinal.day)

===== 참여 모드(스크립트) 종료 =====
승자: mafia | 일수: 2


### 직접 참여하기 (사람 1 + AI 7)

아래 셀은 `input()`으로 직접 입력한다. `?질문`을 입력하면 사회자(룰북 RAG)가 답한다.
`run_with_human`이 interrupt 처리 루프(`invoke -> payload 확인 -> 입력 받기 -> Command(resume)`)를 대신해준다.

In [9]:
# ===== 8. 직접 참여 (input) =====
moderator = agents.Moderator(vs)

pgame = core.new_game(random.sample(config.CHARACTERS, 7), config.HUMAN_NAME, seed=None)
pctrls = agents.build_controllers(pgame, vs)
pctrls[config.HUMAN_NAME] = mg.GraphHumanController(config.HUMAN_NAME)   # input 대신 interrupt 사용

pgraph = mg.build_graph(pctrls)
pfinal = mg.run_with_human(pgraph, pgame, thread_id="human-play", ask_rule=moderator.explain_rule)

print(core.reveal_report(pfinal))


===== 마피아 게임 시작 (8명) =====
참가자: 윤서진, 박준수, 임채원, 최민지, 강태호, 한소율, 오지환, 서지우
  [비밀 · 마피아 서지우] 당신의 직업: 마피아
  [비밀 · 마피아 서지우] 동료 마피아: 한소율

===== 1일차 밤 =====
----------------------------------------
[1일차] 당신은 마피아 입니다.
생존자: 윤서진, 박준수, 임채원, 최민지, 강태호, 한소율, 오지환, 서지우
[비공개] 동료 마피아: 한소율
[동료 제안] {'target': '박준수', 'reason': '박준수는 논리적으로 추리해 주목받는 사람으로 보이며, 시민팀에 위협이 될 수 있는 인물입니다.'}
처치할 대상을 고르세요.
  1. 윤서진
  2. 박준수
  3. 임채원
  4. 최민지
  5. 강태호
  6. 오지환
> ?의사는 자기 자신을 보호할 수 있나요?   ← (자동 응답)
[사회자] 네, 의사는 자기 자신을 보호할 수 있으며, 연속으로 보호할 수도 있습니다.
(룰북 p.3) (룰북 p.4)
> 4   ← (자동 응답)
최민지(시민)이 밤에 사망

===== 1일차 낮 토론 =====
----------------------------------------
[1일차] 당신은 마피아 입니다.
생존자: 윤서진, 박준수, 임채원, 강태호, 한소율, 오지환, 서지우
사망자: 최민지(시민)
[비공개] 동료 마피아: 한소율
[비공개] 1일차 밤 처치 시도: 최민지 (사망)
발언을 입력하세요. (빈 입력은 패스)
> 저는 시민입니다. 오지환님은 투표할 때 이유를 제대로 말하지 않았어요. 설명해 주세요.   ← (자동 응답)
[서지우] 저는 시민입니다. 오지환님은 투표할 때 이유를 제대로 말하지 않았어요. 설명해 주세요.
[박준수] 오지환 씨, 왜 서지우 씨한테 그런 말을 했어요? 뭔가 숨기고 있는 건 아닌가요?
[오지환] 서지우야! 네가 마피아야! 밤 조사 결과를 들었어! 빨리 투표해!
[한소율] 박준수 씨, 서지우 씨를

## 5. 관전 N판 통계

여러 판을 자동으로 돌려 **마피아/시민 승률, 평균 진행 일수, 안전장치 발동 횟수**를 모은다.
LLM으로는 한 판이 수 분 걸리므로 기본 `N_LLM`은 작게 둔다. 먼저 RandomController로 규칙/그래프의 균형을 빠르게 본다.

In [10]:
# ===== 9. 빠른 통계: RandomController 50판 =====
N_RANDOM = 50


def random_factory(game):
    return {p.name: core.RandomController(p.name, seed=i + (game.seed or 0) * 10) for i, p in enumerate(game.players)}


summary_random = mg.run_batch(N_RANDOM, random_factory, seed0=0)
mg.print_batch_summary(summary_random)

===== 관전 50판 통계 =====
마피아팀 승률 : 82.0% (41판)
시민팀 승률   : 18.0% (9판)
무승부        : 0판
평균 진행 일수: 2.82일
안전장치 합계 : 재시도 0, fallback 0, 정체노출 차단 0, AI 결정 요청 2265


In [11]:
# ===== 10. LLM 통계 (시간이 오래 걸림: N_LLM 을 늘리면 더 정확하지만 느리다) =====
N_LLM = 2   # 한 판 수 분. 발표 전 여유가 있을 때만 늘리세요.

t0 = time.time()
summary_llm = mg.run_batch(N_LLM, lambda g: agents.build_controllers(g, vs), seed0=100)
mg.print_batch_summary(summary_llm)
print("총 소요: %.0f초 (판당 %.0f초)" % (time.time() - t0, (time.time() - t0) / N_LLM))

===== 관전 2판 통계 =====
마피아팀 승률 : 50.0% (1판)
시민팀 승률   : 50.0% (1판)
무승부        : 0판
평균 진행 일수: 4.00일
안전장치 합계 : 재시도 0, fallback 0, 정체노출 차단 0, AI 결정 요청 104
총 소요: 523초 (판당 261초)


## 6. 정리

- **State / Node / Edge**: 게임 한 판을 "노드(한 단계) + 엣지(순서)"로 선언해 흐름이 눈에 보인다.
- **조건부 엣지**: 승리 판정, 반복(발언/투표), 동점 재투표를 분기 함수로 표현했다.
- **interrupt + Command(resume)**: 사람 입력을 기다리는 일을 `input()` 루프 없이 그래프 기능으로 처리했다.
- **checkpointer + thread_id**: 노드마다 상태가 저장되어 멈춤/재개가 가능하고, 지나온 경로도 확인할 수 있다.
- 규칙 판정은 모두 `mafia_core`가 하므로 03(GameEngine)과 04(LangGraph)는 **같은 규칙, 다른 진행 방식**이다.